# Week 3.1: Tool Use - Building AI Agents

## Key Insight: Thought → Action → Observation → Answer

Your digital twin learns to USE TOOLS autonomously!

## What You'll Build
- Tools your AI can call (calculator, search, etc.)
- Agent that decides when to use tools
- Multi-step reasoning loops
- Digital twin with tool capabilities

In [1]:
import json
from typing import Dict, List, Any, Callable
from datetime import datetime

print('Week 3.1: Tool Use & Agents')
print('Goal: Make your digital twin DO things, not just talk!')

Week 3.1: Tool Use & Agents
Goal: Make your digital twin DO things, not just talk!


## Part 1: Creating Tools

A tool is a function your AI can call.

In [2]:
# Tool Example 1: Calculator
def calculator(expression: str) -> Dict[str, Any]:
    '''Calculate math expressions.'''
    try:
        result = eval(expression)  # Safe in controlled environment
        return {'result': result, 'expression': expression}
    except Exception as e:
        return {'error': str(e)}

# Tool Example 2: Get current time
def get_time() -> Dict[str, Any]:
    '''Get current time.'''
    now = datetime.now()
    return {
        'time': now.strftime('%H:%M:%S'),
        'date': now.strftime('%Y-%m-%d'),
        'day': now.strftime('%A')
    }

print('Calculator:', calculator('15 * 0.15'))
print('Time:', get_time())

Calculator: {'result': 2.25, 'expression': '15 * 0.15'}
Time: {'time': '05:23:55', 'date': '2026-11-09', 'day': 'Monday'}


## Part 2: Tool Registry

Manage available tools.

In [3]:
class ToolRegistry:
    '''Manages available tools for the agent.'''

    def __init__(self):
        self.tools = {}

    def register(self, name: str, func: Callable, description: str):
        '''Register a tool.'''
        self.tools[name] = {
            'function': func,
            'description': description,
            'name': name
        }

    def get_tool(self, name: str) -> Callable:
        '''Get tool function by name.'''
        return self.tools[name]['function']

    def list_tools(self) -> str:
        '''Get formatted list of available tools.'''
        lines = ['Available tools:']
        for name, info in self.tools.items():
            lines.append(f"- {name}: {info['description']}")
        return '\n'.join(lines)

#Create registry
registry = ToolRegistry()
registry.register('calculator', calculator, 'Calculate math expressions')
registry.register('get_time', get_time, 'Get current date and time')

print(registry.list_tools())

Available tools:
- calculator: Calculate math expressions
- get_time: Get current date and time


## Part 3: ReAct Agent

Agent that can reason and act with tools.

**ReAct Loop:**
1. **Thought**: Decide what to do
2. **Action**: Use a tool
3. **Observation**: See result
4. **Answer**: Respond to user (or loop again)

In [7]:
class SimpleReActAgent:
    '''Simple agent that can use tools.'''

    def __init__(self, tool_registry: ToolRegistry):
        self.registry = tool_registry
        self.history = []

    def think(self, query: str) -> str:
        '''Decide what action to take.'''
        # Simplified: pattern matching for demo
        query_lower = query.lower()

        if any(word in query_lower for word in ['calculate', 'math', '+', '-', '*', '/']):
            return 'use_calculator'
        elif any(word in query_lower for word in ['time', 'date', 'day']):
            return 'use_time'
        else:
            return 'direct_answer'

    def act(self, action: str, query: str) -> Dict[str, Any]:
        '''Execute the action.'''
        if action == 'use_calculator':
            # Extract math expression from query
            import re
            match = re.search(r'[\d+\-*/().\s]+', query)
            if match:
                expr = match.group(0).strip()
                return self.registry.get_tool('calculator')(expr)
            return {'error': 'Could not extract expression'}

        elif action == 'use_time':
            return self.registry.get_tool('get_time')()

        else:
            return {'direct': query}

    def run(self, query: str) -> str:
        '''Run the agent on a query.'''
        # Step 1: Thought
        thought = self.think(query)
        self.history.append({'step': 'thought', 'content': thought})

        # Step 2: Action
        observation = self.act(thought, query)
        self.history.append({'step': 'action', 'content': observation})

        # Step 3: Answer
        if 'result' in observation:
            answer = f"The result is {observation['result']}"
        elif 'time' in observation:
            answer = f"It's {observation['time']} on {observation['date']} ({observation['day']})"
        elif 'error' in observation:
            answer = f"Error: {observation['error']}"
        else:
            answer = "I can help with calculations and time queries."

        return answer

# Test the agent
agent = SimpleReActAgent(registry)

print('\nAgent Tests:')
print('Q: What is 25 * 4?')
print('A:', agent.run('What is 25 * 4?'))
print('\nQ: What time is it?')
print('A:', agent.run('What time is it?'))


Agent Tests:
Q: What is 25 * 4?
A: Error: invalid syntax (<string>, line 0)

Q: What time is it?
A: It's 05:31:27 on 2026-11-09 (Monday)


## Part 4: Integrating with Digital Twin

Add tools to your Week 1 digital twin.

In [5]:
class DigitalTwinWithTools:
    '''Digital twin that can use tools.'''

    def __init__(self, name: str):
        self.name = name
        self.registry = ToolRegistry()
        self.agent = None
        self.setup_tools()

    def setup_tools(self):
        '''Setup default tools.'''
        self.registry.register('calculator', calculator, 'Calculate math')
        self.registry.register('get_time', get_time, 'Get current time')
        self.agent = SimpleReActAgent(self.registry)

    def add_tool(self, name: str, func: Callable, description: str):
        '''Add a new tool.'''
        self.registry.register(name, func, description)

    def chat(self, message: str) -> str:
        '''Chat with tool use.'''
        return self.agent.run(message)

# Create digital twin with tools
twin = DigitalTwinWithTools('My Assistant')

print('\nDigital Twin with Tools:')
print(twin.chat('Calculate 47.50 * 0.15'))
print(twin.chat('What day is today?'))


Digital Twin with Tools:
The result is 7.125
It's 05:23:55 on 2026-11-09 (Monday)


## Exercise 1: Create Custom Tools

**Task**: Add 2 custom tools to your digital twin.

Ideas:
- Memory search
- Note taker
- Task list
- Habit tracker

In [6]:
# YOUR CODE HERE

def my_custom_tool(arg: str) -> Dict[str, Any]:
    '''Your tool description.'''
    # Implement your tool
    return {'result': 'your result'}

# Add to twin
# twin.add_tool('my_tool', my_custom_tool, 'Description')
# Test: twin.chat('query that uses your tool')

## Summary

### What We Learned:
- Tools are functions AI can call
- Tool registry manages available tools
- ReAct pattern: Think → Act → Observe
- Agents can use tools autonomously
- Digital twin + tools = powerful assistant

### Next: Week 3.2 - Testing Your AI

In [8]:
# Custom Digital Twin with Personalized Memory and Task tools

class EnhancedDigitalTwin(DigitalTwinWithTools):
    def __init__(self, name: str):
        super().__init__(name)
        self.memory = []
        self.tasks = []
        self.setup_custom_tools()

    def setup_custom_tools(self):
        # 1. Custom Memory Search Tool
        def memory_search(query: str) -> dict:
            '''Search personal memories/notes.'''
            matches = [m for m in self.memory if query.lower() in m.lower()]
            return {'results': matches if matches else ["No matches found."]}

        # 2. Custom Task List / Note Taker Tool
        def manage_tasks(action_and_task: str) -> dict:
            '''Manage your task list. Format: 'add <task>' or 'view'.'''
            parts = action_and_task.strip().split(' ', 1)
            action = parts[0].lower()

            if action == 'add' and len(parts) > 1:
                task = parts[1]
                self.tasks.append(task)
                return {'status': f"Added task: {task}"}
            elif action == 'view':
                return {'tasks': self.tasks if self.tasks else ["No tasks found."]}
            else:
                return {'error': "Invalid action. Use 'add <task>' or 'view'"}

        # Registering tools
        self.add_tool('memory_search', memory_search, 'Search through saved memories/facts.')
        self.add_tool('manage_tasks', manage_tasks, 'Add or view things on your task list.')

    # Override chat to support simple custom tool routing patterns
    def chat(self, message: str) -> str:
        msg_lower = message.lower()
        if 'search' in msg_lower or 'remember' in msg_lower:
            # Simulate memory search routing
            search_query = message.replace('search', '').replace('remember', '').strip()
            res = self.registry.get_tool('memory_search')(search_query)
            return f"[Memory Search Result]: {', '.join(res['results'])}"
        elif 'task' in msg_lower or 'todo' in msg_lower:
            if 'add' in msg_lower:
                task_content = message.split('add', 1)[1].strip()
                res = self.registry.get_tool('manage_tasks')(f"add {task_content}")
                return f"[Tasks Manager]: {res['status']}"
            else:
                res = self.registry.get_tool('manage_tasks')('view')
                return f"[Tasks Manager]: Your tasks: {', '.join(res['tasks'])}"
        else:
            return super().chat(message)

# Instantiate and showcase the custom Digital Twin
twin = EnhancedDigitalTwin('Twin-V2')
twin.memory.extend([
    "Favorite programming language is Python.",
    "Coffee preference: Black, no sugar."
])

print("--- Custom Digital Twin Demo ---")
print(twin.chat("search coffee"))
print(twin.chat("add buy groceries to my task list"))
print(twin.chat("view tasks"))


--- Custom Digital Twin Demo ---
[Memory Search Result]: Coffee preference: Black, no sugar.
[Tasks Manager]: Added task: buy groceries to my task list
[Tasks Manager]: Your tasks: buy groceries to my task list
